In [2]:
import sys
project_path = "/home/aj/redesigned-octo-couscous" 
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import dct

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = AutoModelForCausalLM.from_pretrained(model_name, _attn_implementation="eager").to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left", truncation_side="left")
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [3]:
# source_layer, target_layer = 7, 13
source_layer, target_layer = 10, 20

sliced_model = dct.SlicedModel(model, source_layer, target_layer, layers_name="model.layers")

In [4]:
import pandas as pd

dataset = pd.read_csv(project_path + "/harmful_behaviors.csv")
instructions = dataset["target"].tolist()

_examples = instructions[:250]
# examples = _examples

chat_init = [{'content':"You are a helpful assistant", 'role':'system'}]
chats = [chat_init + [{'content': content, 'role':'user'}] for content in _examples]
examples = [tokenizer.apply_chat_template(chat, add_special_tokens=False, tokenize=False, add_generation_prompt=True) for chat in chats]

In [5]:
from tqdm.notebook import tqdm

d_model = model.config.hidden_size
n_samples = len(examples)
seq_len = 27
fwd_batch_size = 1

X = torch.zeros(n_samples, seq_len, d_model, device=device, dtype=model.dtype)
Y = torch.zeros(n_samples, seq_len, d_model, device=device, dtype=model.dtype)

for t in tqdm(range(0, n_samples, fwd_batch_size)):
    with torch.no_grad():
        model_inputs = tokenizer(examples[t:t+fwd_batch_size], return_tensors="pt", truncation=True, padding="max_length", max_length=seq_len).to(device)
        hidden_states = model(model_inputs["input_ids"], output_hidden_states=True).hidden_states
        h_source = hidden_states[source_layer] # b x t x d_model
        unsteered_target = sliced_model(h_source) # b x t x d_model

        X[t:t+fwd_batch_size, :, :] = h_source
        Y[t:t+fwd_batch_size, :, :] = unsteered_target

  0%|          | 0/250 [00:00<?, ?it/s]

In [6]:
from torch import vmap

factor_batch_size = 256

delta_acts_single = dct.DeltaActivations(sliced_model, slice(-3, None)).to(device)
delta_acts = vmap(delta_acts_single, in_dims=(1,None,None), out_dims=2,
                  chunk_size=factor_batch_size)

In [7]:
steering_calibrator = dct.SteeringCalibrator(target_ratio=.5)
input_scale = steering_calibrator.calibrate(delta_acts_single, X, Y, factor_batch_size=factor_batch_size)

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

In [8]:
input_scale

9.466666698455356

In [9]:
X, Y = X.to(device), Y.to(device)

num_factors = 256
form = "exp"

if form == "lin":
    dct_ = dct.LinearDCT(num_factors=num_factors)
    U,V = dct_.fit(delta_acts_single, X, Y, input_scale=input_scale, factor_batch_size=factor_batch_size)

if form == "quad":
    dct_ = dct.QuadraticDCT(num_factors=num_factors)
    U,V = dct_.fit(delta_acts_single, X, Y, factor_batch_size=factor_batch_size)
    
if form == "exp":
    dct_ = dct.ExponentialDCT(num_factors=num_factors)
    U,V = dct_.fit(delta_acts_single, X, Y, input_scale=input_scale, factor_batch_size=factor_batch_size, init="random")

initializing V,U...
training...


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

...

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

In [10]:
# slice_to_end = dct.SlicedModel(
#     model,
#     start_layer=source_layer,
#     end_layer=model.config.num_hidden_layers,
#     apply_final_norm=True,
# )
# delta_acts_end_single = dct.DeltaActivations(slice_to_end)

# Y_end = torch.zeros_like(Y)
# with torch.no_grad():
#     for b in range(0, X.shape[0], fwd_batch_size):
#         Y_end[b:b+fwd_batch_size] = slice_to_end(X[b:b+fwd_batch_size])

# REFUSAL_TOKEN = tokenizer.encode("I", add_special_tokens=False)[0]
# SURE_TOKEN = tokenizer.encode("Sure", add_special_tokens=False)[0]
# with torch.no_grad():
#     target_vec = model.lm_head.weight.data[SURE_TOKEN,:] - model.lm_head.weight.data[REFUSAL_TOKEN,:]

# scores, indices = dct_.rank(delta_acts_end_single, X, Y_end, target_vec=target_vec,
#                             batch_size=fwd_batch_size, factor_batch_size=factor_batch_size)

# pd.DataFrame({"indices": indices.cpu().numpy(), "scores": scores.cpu().numpy()})

In [11]:
model_editor = dct.ModelEditor(model, layers_name="model.layers")

In [57]:
import importlib
import matplotlib.pyplot as plt
import numpy as np
import dct_attrib
importlib.reload(dct_attrib)
from dct_attrib import DCTAttrib

batch_size = 100000

attrib = DCTAttrib(V, U, device)

In [13]:
def plot_top_three_circuit_diversity():
    circuit_groups = []
    for target_factor in tqdm(range(U.shape[1]), desc="Attributing factors"):
        with torch.no_grad():
            top_circuits, _ = attrib.I(
                target_factor,
                width=3,
                k=3,
                batch_size=batch_size,
                input_scale=input_scale,
                silent=True,
            )
        circuits, _ = top_circuits.get()
        circuit_groups.append(circuits.cpu().numpy())

    num_groups = len(circuit_groups)
    diversity = np.zeros((num_groups, num_groups))
    for left in range(num_groups):
        for right in range(left + 1, num_groups):
            pairwise_distances = [
                1 - len(set(left_circuit) & set(right_circuit)) / len(
                    set(left_circuit) | set(right_circuit)
                )
                for left_circuit in circuit_groups[left]
                for right_circuit in circuit_groups[right]
            ]
            diversity[left, right] = diversity[right, left] = np.mean(pairwise_distances)

    total_pairwise_diversity = diversity[np.triu_indices(num_groups, k=1)].sum()

    fig, axis = plt.subplots(constrained_layout=True)
    image = axis.imshow(diversity, vmin=0, vmax=1)
    axis.set(
        xlabel="Other target factor",
        ylabel="Target factor",
    )
    fig.colorbar(image, ax=axis, label="Mean Jaccard distance")
    plt.show()

    return total_pairwise_diversity / (num_groups * (num_groups - 1) / 2)

# plot_top_three_circuit_diversity()

In [14]:
with torch.no_grad():
    top_circuits, _ = attrib.I(
        0,
        width=3,
        k=1,
        batch_size=batch_size,
        input_scale=1,
        silent=True,
    )
    circuit_indices, k = top_circuits.get()
print(circuit_indices, k)

tensor([[ 59, 127, 153]], device='cuda:0', dtype=torch.int16) tensor([1.8870])
